# Model Preflight (Colab)

Verifies that VideoMAE-Base and Video Swin-Tiny construct, load pretrained weights, accept the canonical input, and return correctly shaped logits on a real GPU.

`docs/MODEL_CONTRACT.md` requires preflight to pass before any real training run.

This notebook is a launcher. It contains no model, dataset, or training logic — those live in `src/` and `scripts/`, per the notebook boundary in `docs/ARCHITECTURE.md`.

**Runtime → Change runtime type → GPU** before running.

## 1. Get the code

In [ ]:
import os

REPO_URL = "https://github.com/Adgonzalez2018/ASL-Recognition-Model.git"

!git clone -q $REPO_URL /content/asl
PROJECT = "/content/asl/ASL_training"

assert os.path.isdir(PROJECT), f"project not found at {PROJECT}"
print(PROJECT)

## 2. Install

Colab ships a torch build matched to its CUDA driver. Installing with `--no-deps` avoids replacing it, which is slow and a common source of driver mismatch. See `docs/ENVIRONMENTS.md`.

In [ ]:
!pip install -q -e "$PROJECT" --no-deps
!pip install -q "transformers>=4.44" pyyaml pandas scikit-learn av tqdm

## 3. Record the environment

Every real run must capture this, per `docs/TRAINING_CONTRACT.md`. Colab may assign a different GPU on each session.

In [ ]:
import platform

import torch
import torchvision
import transformers

print(f"python       {platform.python_version()}")
print(f"torch        {torch.__version__}")
print(f"torchvision  {torchvision.__version__}")
print(f"transformers {transformers.__version__}")
print(f"cuda         {torch.version.cuda}")
print(f"gpu          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

if not torch.cuda.is_available():
    print("\nNo GPU. Runtime -> Change runtime type -> GPU.")

## 4. Preflight both architectures

`--num-classes 2731` is the full ASL Citizen vocabulary. Once the data layer exists (Phase 2), this value comes from the generated label map instead of being typed here.

In [ ]:
NUM_CLASSES = 2731
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

!cd "$PROJECT" && python scripts/model_preflight.py \
    --config configs/models/videomae_base.yaml \
    --num-classes $NUM_CLASSES --device $DEVICE --batch-size 2

In [ ]:
!cd "$PROJECT" && python scripts/model_preflight.py \
    --config configs/models/video_swin_tiny.yaml \
    --num-classes $NUM_CLASSES --device $DEVICE --batch-size 2

## 5. Find a workable batch size

Probes peak memory per architecture. Useful for planning, but note that these are activations for a forward and backward pass only — a real run also holds optimizer state.

A batch size chosen on an A100 will fail on a T4, and Colab may give you either.

In [ ]:
import json
import subprocess

if torch.cuda.is_available():
    for config in ["videomae_base", "video_swin_tiny"]:
        for batch_size in [2, 4, 8, 16]:
            result = subprocess.run(
                [
                    "python",
                    "scripts/model_preflight.py",
                    "--config",
                    f"configs/models/{config}.yaml",
                    "--num-classes",
                    str(NUM_CLASSES),
                    "--device",
                    "cuda",
                    "--batch-size",
                    str(batch_size),
                    "--json",
                ],
                cwd=PROJECT,
                capture_output=True,
                text=True,
            )
            try:
                report = json.loads(result.stdout)
            except json.JSONDecodeError:
                print(f"{config:18} batch {batch_size:3}  could not parse report")
                continue

            if report.get("status") == "passed":
                print(f"{config:18} batch {batch_size:3}  {report['peak_memory_mb']:>9} MB")
            else:
                print(f"{config:18} batch {batch_size:3}  FAILED: {report.get('error', '')[:60]}")
                break

## Notes

Preflight is a structural check on synthetic tensors. It establishes that the software path works. It says nothing about model quality, and is not an experiment.

Next: Phase 2, the ASL Citizen audit and data layer. See `docs/CURRENT_PHASE.md`.